<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.1-setup/notebooks/GCP_Capstone_1.1_Setup.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.1 Setting Up Your GCP AI Project
**Netsetos GenAI Engineering — GCP Capstone**

This notebook verifies your GCP project setup and runs your first Gemini calls.


## Step 1: Install Dependencies


In [ ]:
!pip install -q google-genai google-cloud-aiplatform google-cloud-firestore google-cloud-storage


## Step 2: Authenticate (if running on Colab)


In [ ]:
# Only needed on Google Colab (not Cloud Shell or Workbench)
import sys
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()


## Step 3: Set Your Project


In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"  # <-- CHANGE THIS
LOCATION = "us-central1"

!gcloud config set project {PROJECT_ID}
print(f"Project: {PROJECT_ID}")


## Step 4: Enable & Verify APIs

In [ ]:
# GCP APIs are OFF by default (a security feature) — enable everything the
# capstone needs. Idempotent: safe to re-run. Freshly-enabled APIs can take
# up to ~60s to propagate, so if your first Gemini call (Step 6) returns
# 403 SERVICE_DISABLED, wait a minute and re-run it.
CAPSTONE_APIS = [
    "aiplatform.googleapis.com",       # Vertex AI — Gemini, embeddings, tuning
    "run.googleapis.com",              # Cloud Run
    "firestore.googleapis.com",        # Firestore — vector search + metadata
    "cloudbuild.googleapis.com",       # Cloud Build
    "secretmanager.googleapis.com",    # Secret Manager
    "storage.googleapis.com",          # Cloud Storage
    "documentai.googleapis.com",       # Document AI — OCR / layout
    "speech.googleapis.com",           # Speech-to-Text
    "vision.googleapis.com",           # Vision
    "language.googleapis.com",         # Natural Language
    "translate.googleapis.com",        # Translation
    "bigquery.googleapis.com",         # BigQuery
    "artifactregistry.googleapis.com", # Artifact Registry
    "monitoring.googleapis.com",       # Cloud Monitoring
    "logging.googleapis.com",          # Cloud Logging
]
print(f"Enabling {len(CAPSTONE_APIS)} APIs on {PROJECT_ID} (~30-60s)...")
!gcloud services enable {" ".join(CAPSTONE_APIS)} --project={PROJECT_ID}

# Verify — the critical ones must be ACTIVE before Step 6 will work
import subprocess
result = subprocess.run(
    ["gcloud", "services", "list", "--enabled", "--format=value(name)"],
    capture_output=True, text=True,
)
enabled = set(result.stdout.split())
print(f"\n✅ {len(enabled)} APIs enabled on {PROJECT_ID}")
for api in CAPSTONE_APIS:
    ok = api in enabled
    print(f"  {'✅' if ok else '⏳ still propagating — wait ~60s and re-run'} {api.split('.')[0]}")

## Step 5: Initialize Gemini Client


In [ ]:
from google import genai

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
print("✅ Gemini client initialized")


## Step 6: First Gemini Call + Token Usage


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Explain what Google Cloud Platform is in exactly 2 sentences.'
)

print('✅ Gemini says:')
print(response.text)

# Token usage
usage = response.usage_metadata
print(f'\n📊 Token Usage:')
print(f'  Input tokens:    {usage.prompt_token_count}')
print(f'  Output tokens:   {usage.candidates_token_count}')
print(f'  Thinking tokens: {getattr(usage, "thoughts_token_count", 0)}')
print(f'  Total tokens:    {usage.total_token_count}')

# Calculate cost (Gemini 3.6 Flash STANDARD: $1.50/M in, $7.50/M out; intro promo ~half thru Dec 31 2026)
USD_TO_INR = 85
cost = (usage.prompt_token_count * 1.50 + usage.candidates_token_count * 7.50) / 1_000_000
print(f'  Cost: ${cost:.6f} (₹{cost*USD_TO_INR:.4f})')
print(f'\n  With $500: {500/cost:,.0f} calls possible')
print('\n🎉 Setup complete! Your GCP GenAI environment is ready.')


## Step 7: Free Token Counter


In [ ]:
# count_tokens is FREE — no inference, no cost
texts = [
    'Hello',
    'Hyderabad is the capital of Telangana' * 10,
    'The transformer architecture uses self-attention' * 100,
]

print('📈 Token Counts (FREE — no API cost):')
for t in texts:
    result = client.models.count_tokens(model='gemini-3.6-flash', contents=t)
    est_cost = result.total_tokens * 1.50 / 1_000_000 * USD_TO_INR
    print(f'  {result.total_tokens:>6,} tokens | {len(t):>6} chars | ₹{est_cost:.4f} input | {t[:40]}...')


## Step 8: Compare 3 Models


In [ ]:
import time

prompt = 'Explain the difference between SQL and NoSQL databases. Give 2 examples of each.'
models = [
    ('gemini-3.1-flash-lite', 0.25, 1.50),
    ('gemini-3.6-flash', 1.50, 7.50),
    ('gemini-3.1-pro-preview', 2.00, 12.00),
]

print(f"{'Model':<30} {'Tokens':>7} {'Latency':>9} {'Rs':>9}")
print('-' * 60)
for name, ip, op in models:
    t0 = time.time()
    try:
        r = client.models.generate_content(model=name, contents=prompt)
        ms = (time.time()-t0)*1000
        u = r.usage_metadata
        cost = (u.prompt_token_count*ip + u.candidates_token_count*op)/1e6*85
        print(f'{name:<30} {u.total_token_count:>7} {ms:>7.0f}ms Rs.{cost:>6.4f}')
    except Exception as e:
        print(f'{name:<30} ERROR: {e}')


## ✅ Lesson 1.1 Complete!

- ✅ GCP project created and configured
- ✅ Billing linked with budget alerts
- ✅ 15+ APIs enabled
- ✅ Gemini client initialized with `google-genai`
- ✅ First Gemini call with token usage and cost
- ✅ Free token counter tested
- ✅ 3 models compared (Flash-Lite, Flash, Pro)

**Next: Lesson 1.2 — IAM & Security for GenAI Projects**
